<center><h1>Mutyala_Yogita_HW3</h1></center>
<br>
<br>

Name: Yogita Mutyala
<br>
Github Username: YogitaMutyala
<br>
USC ID: 4421997071

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import bootstrap

Get the AReM Data Set

In [2]:
arem_data_path = Path("../data/AReM")
activities = ["bending1", "bending2", "cycling", "lying","sitting", "standing", "walking"]

Understanding: The AReM dataset contains 88 instances from seven human activities: bending1, bending2, cycling, lying, sitting, standing and walking

Each instance contains six time series with 480 observations:

- avg_rss12
- var_rss12
- avg_rss13
- var_rss13
- avg_rss23
- var_rss23

### (b) Test and Train Data

In [3]:
train_datasets, test_datasets = [], []
# create list of both activity labels and paths for each dataset file
for a in activities:
    for i in sorted((arem_data_path / a).glob("dataset*.csv")):
        number = int(i.stem.replace("dataset", ""))
        test_numbers = [1, 2] if a.startswith("bending") else [1, 2, 3]
        (test_datasets if number in test_numbers else train_datasets).append((a, i))
#check
print("Training instances:", len(train_datasets))
print("Test instances:", len(test_datasets))

Training instances: 69
Test instances: 19


Understanding: Datasets 1 and 2 from both bending1 and bending2 are used as test data. Datasets 1, 2, and 3 from each of the remaining activities are also used as test data. All other datasets are used as training data.


### (c) Feature Extraction

#### i. Research

Types of time-domain features used in time series classification:

- Minimum 
- Maximum
- Mean
- Median
- Range
- Median absolute deviation
- Root mean square
- Variance
- Standard deviation
- First Quartile
- Third quartiles
- Interquartile range
- Skewness
- Autocorrelation
- Number and location of peaks

#### ii. Extraction

In [4]:
ts_columns = ["time", "avg_rss12", "var_rss12", "avg_rss13","var_rss13", "avg_rss23", "var_rss23"]

def extract_ft(file):
    sep = r"\s+" if file.parent.name == "bending2" and file.stem == "dataset4" else ","
    data = pd.read_csv(file, comment="#", header=None, names=ts_columns,sep=sep, usecols=range(7), engine="python")
    features = {}
    for i, col in enumerate(ts_columns[1:], 1):
        x = data[col]
        features.update({
            f"min{i}": x.min(),
            f"max{i}": x.max(),
            f"mean{i}": x.mean(),
            f"median{i}": x.median(),
            f"std{i}": x.std(),
            f"1st_quart{i}": x.quantile(0.25),
            f"3rd_quart{i}": x.quantile(0.75)
        })
    return features
rows = []
for split, files in [("train", train_datasets), ("test", test_datasets)]:
    for activity, file in files:
        rows.append({
            "activity": activity,
            "dataset": file.stem,
            "split": split,
            **extract_ft(file)
        })
features_df = pd.DataFrame(rows)
print("Instances:", len(features_df))
print("Extracted features:", features_df.shape[1] - 3)
features_df.head()

Instances: 88
Extracted features: 42


,activity,dataset,split,min1,max1,mean1,median1,std1,1st_quart1,3rd_quart1,...,std5,1st_quart5,3rd_quart5,min6,max6,mean6,median6,std6,1st_quart6,3rd_quart6
0,bending1,dataset3,train,35.00,47.40,43.954500,44.33,1.558835,43.00,45.00,...,1.999604,35.3625,36.50,0.0,1.79,0.493292,0.43,0.513506,0.00,0.94
1,bending1,dataset4,train,33.00,47.75,42.179812,43.50,3.670666,39.15,45.00,...,3.849448,30.4575,36.33,0.0,2.18,0.613521,0.50,0.524317,0.00,1.00
2,bending1,dataset5,train,33.00,45.75,41.678063,41.75,2.243490,41.33,42.75,...,2.411026,28.4575,31.25,0.0,1.79,0.383292,0.43,0.389164,0.00,0.50
3,bending1,dataset6,train,37.00,48.00,43.454958,43.25,1.386098,42.50,45.00,...,2.488862,22.2500,24.00,0.0,5.26,0.679646,0.50,0.622534,0.43,0.87
4,bending1,dataset7,train,36.25,48.00,43.969125,44.50,1.618364,43.31,44.67,...,3.318301,20.5000,23.75,0.0,2.96,0.555312,0.49,0.487826,0.00,0.83


Understanding: This gave 42 features for each instance. Activity,dataset,split are excluded columns - identifying columns

#### iii. Standard Deviation

In [5]:
feature_cols = features_df.columns[3:]
results = []
# find sd for each of the 42 features with 90% bootstrap confidence level
for col in feature_cols:
    x = features_df[col].dropna().to_numpy()
    ci = bootstrap(
        (x,), lambda y: np.std(y, ddof=1),
        confidence_level=0.90, # 90% confidence level
        n_resamples=1000,
        method="percentile",
        random_state=42
    ).confidence_interval
    results.append([col, x.std(ddof=1), ci.low, ci.high])
std_results = pd.DataFrame( results, columns=["Feature", "Standard Deviation", "90% CI Lower", "90% CI Upper"])
std_results

,Feature,Standard Deviation,90% CI Lower,90% CI Upper
0,min1,9.569975,8.275241,10.747623
1,max1,4.394362,3.343823,5.296316
2,mean1,5.335718,4.686470,5.869648
3,median1,5.440054,4.788491,6.016169
4,std1,1.772153,1.569967,1.945903
5,1st_quart1,6.153590,5.547790,6.646684
6,3rd_quart1,5.138925,4.330721,5.825098
7,min2,0.000000,0.000000,0.000000
8,max2,5.062729,4.645742,5.403547
9,mean2,1.574164,1.411542,1.710131


Understanding: 

min1 has the greatest variation with a standard deviation of 9.57 and 90% confidence interval of [8.28, 10.75]. 

min2 and min4 have standard deviations and confidence intervals of zero because their minimum values are the same across all instances. 

Overall, most confidence intervals are reasonably narrow indicating that the estimated standard deviations are stable

#### iv. Select Features

I think the three most important time domain features are  minimum, mean, and maximum

Minimum and maximum capture the extreme sensor values reached during activity and mean represents the overall sensor level.

## 2. ISLR 3.7.4

### (a) Linear Train

The training RSS of cubic regression will be less than or equal to the that of the linear regression. The cubic model contains the linear model as well as additional predictors X2 and X3.

The model has more flexibility to fit the training data, so adding these predictors will not increase the training RSS.

### (b) Linear Test

The linear regression is expected to have a lower test RSS because  the true relationship is linear. The cubic model may fit random noise in the training data that will lead to overfitting and higher test RSS.

### (c) Not Linear Train

Training RSS of cubic regression will have a less than or equal to that of linear regression. This is because the cubic model is more flexible and it includes the linear model as a special case, regardless of the true relationship

### (d) Not Linear Testing

In this case, I think there isn't enough information to determine which model will have a lower test RSS. If the true relationship is close to cubic then the cubic model will perform better. The linear model may perform better if it is slightly non linear or has a different form because it has lower variance and is less likely to overfit.